# Export time series theo session — cho dashboard cá nhân (chọn ngày)

Xuất dữ liệu chuỗi thời gian (không phải số tổng hợp theo phase) cho từng
session, để dashboard cá nhân hiển thị khi user chọn 1 ngày cụ thể:

1. **EEG band power theo thời gian** — từ `features_per_epoch.csv` (epoch 2s),
   5 băng chính (alpha/theta/beta/gamma/high_beta, tương đối, TB AF3+AF4),
   downsample về tối đa ~60 điểm/băng/session để giữ file nhẹ.
2. **HR/RMSSD toàn phiên** — ghép REST_BEFORE (1 điểm) + 21 cửa sổ ~2 phút
   trong MEDITATION (`ppg_hrv_wide_by_session.csv`, đã có sẵn — REST không
   chia cửa sổ thêm vì quá ngắn, theo đúng thiết kế gốc của
   `ppg_hrv_batch_analysis.ipynb`) + REST_AFTER (1 điểm). Tính max/min trên
   toàn bộ chuỗi này.
3. **HbO/HbR theo thời gian** — từ `layer2/.../fnirs_hbo_hbr.csv` (100Hz,
   bản v1 CHƯA motion-correction — dùng cho trực quan hoá, không dùng để
   tính chỉ số thống kê chính thức), downsample về ~150 điểm/session.

**Output**: `outputs/session_timeseries.json` — nạp trực tiếp vào dashboard.

In [ ]:
from pathlib import Path
import json
import re
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)

REPO_ROOT = Path("../..").resolve()
ANALYTICS_DIR = REPO_ROOT / "data" / "analytics"
LAYER2_DIR = ANALYTICS_DIR / "layer2"
LIB_DIR = REPO_ROOT / "lib"
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))
from within_session_dynamics import detect_drowsy_episodes
OUT_DIR = Path(".").resolve() / "outputs"
OUT_DIR.mkdir(exist_ok=True)

PHASE_ORDER = ["REST_BEFORE", "MEDITATION", "REST_AFTER"]
BANDS = ["delta", "theta", "alpha", "beta", "gamma"]  # 5 bang co dien, lay ABSOLUTE power (V^2) theo yeu cau
EPOCH_SEC = 2.0

eeg_epoch = pd.read_csv(ANALYTICS_DIR / "eeg_features" / "features_per_epoch.csv")
ppg_wide = pd.read_csv(ANALYTICS_DIR / "ppg_hrv" / "ppg_hrv_wide_by_session.csv")
fnirs_phase = pd.read_csv(ANALYTICS_DIR / "fnirs_hemo_v2" / "fnirs_hemo_by_phase_v2.csv")
# baseline/latency da validate (median+1SD, xem eeg_within_session_dynamics.ipynb muc 2c) --
# dung de ve diem "bat dau thu gian" tren chuoi thoi gian alpha_power_rel moi session.
onset_lookup = pd.read_csv(ANALYTICS_DIR / "eeg_dynamics" / "within_session_dynamics_by_session.csv").set_index("File")

session_id_lookup = (
    fnirs_phase[["participant_key", "study_day", "session_id"]]
    .drop_duplicates()
    .set_index(["participant_key", "study_day"])["session_id"]
    .to_dict()
)

# Gio bat dau phien (GMT+7, da quy doi san o Layer 1) -- de dashboard ca nhan hien thi
# "thuc hien luc may gio" cho tung ngay, chi hien thi thong tin, khong dung de tinh toan gi.
# Uu tien session_role=primary_valid neu 1 (participant, ngay) co nhieu ban ghi.
layer1_sessions = pd.read_csv(ANALYTICS_DIR / "layer1" / "sessions.csv")
layer1_sessions["_role_rank"] = (layer1_sessions["session_role"] == "primary_valid").astype(int)
start_time_lookup = (
    layer1_sessions.dropna(subset=["participant_key", "start_time_gmt7"])
    .sort_values("_role_rank", ascending=False)
    .drop_duplicates(subset=["participant_key", "study_day"], keep="first")
    .set_index(["participant_key", "study_day"])["start_time_gmt7"]
    .to_dict()
)

sessions_scope = (
    eeg_epoch[["participant_key", "study_day", "File"]]
    .drop_duplicates()
    .sort_values(["participant_key", "study_day"])
)
print(f"{len(sessions_scope)} session co EEG epoch data")
print(f"{len(onset_lookup)} session co onset latency da tinh")
print(f"{len(start_time_lookup)} session co gio bat dau")

## 1. EEG band power theo thời gian (downsample bin trung bình)

5 băng cổ điển đầy đủ **delta/theta/alpha/beta/gamma**, dùng **absolute
power** (tích phân PSD trong băng, quy đổi sang **µV²** = ×1e12 từ V² gốc —
xem `metadata_multimodal/README.md` mục EEG — để tránh làm tròn về 0 và dễ
đọc hơn theo chuẩn tài liệu EEG). Lưu ý: delta thường có biên độ lớn hơn hẳn
các băng cao (do 1/f trong phổ EEG) — biểu đồ nhiều đường trên cùng 1 trục y
sẽ bị delta lấn át về mặt hình ảnh, đây là đặc điểm thật của tín hiệu chứ
không phải lỗi.

In [ ]:
def downsample_bin(values, max_points):
    n = len(values)
    if n <= max_points:
        return values
    bin_size = int(np.ceil(n / max_points))
    n_bins = int(np.ceil(n / bin_size))
    out = []
    for i in range(n_bins):
        chunk = values[i * bin_size:(i + 1) * bin_size]
        out.append(float(np.nanmean(chunk)))
    return out

def extract_eeg_timeseries(participant_key, study_day, file_name, max_points=60):
    sub = eeg_epoch[
        (eeg_epoch["participant_key"] == participant_key) &
        (eeg_epoch["study_day"] == study_day) &
        (eeg_epoch["File"] == file_name)
    ].copy()
    if sub.empty:
        return None
    sub["phase_order"] = sub["phase"].map({p: i for i, p in enumerate(PHASE_ORDER)}).fillna(99)
    sub = sub.sort_values(["phase_order", "epoch_idx"]).reset_index(drop=True)

    out = {"n_epochs": len(sub)}
    phase_seq = sub["phase"].tolist()
    out["phase_bounds"] = downsample_bin([PHASE_ORDER.index(p) if p in PHASE_ORDER else -1 for p in phase_seq], max_points)
    for band in BANDS:
        col_af3, col_af4 = f"AF3_{band}_power_abs", f"AF4_{band}_power_abs"
        if col_af3 in sub.columns and col_af4 in sub.columns:
            # V^2 -> uV^2 (x1e12) -- gia tri V^2 tho (~1e-11) se lam tron thanh 0.0
            # o 4 chu so thap phan; uV^2 la don vi pho bien hon trong tai lieu EEG.
            mean_band = (sub[[col_af3, col_af4]].mean(axis=1) * 1e12).tolist()
            out[band] = [round(v, 3) for v in downsample_bin(mean_band, max_points)]
    return out

print(extract_eeg_timeseries(*sessions_scope.iloc[0]).keys())

## 1b. Thời điểm bắt đầu thư giãn (relax onset) — cùng đơn vị/ngưỡng với
`eeg_within_session_dynamics.ipynb` (alpha_power_rel, KHÔNG phải absolute
power như mục 1 ở trên — 2 đại lượng khác nhau, không dùng chung 1 trục y).
Trục thời gian tính theo "giây kể từ epoch đầu MEDITATION còn giữ lại"
(giống hệt cách `onset_latency_sec` được tính — bỏ qua khoảng trống do
epoch bị loại QC ở giữa, xem `lib/within_session_dynamics.py`).

In [ ]:
ONSET_SMOOTH_WIN = 3  # phai khop chinh xac SMOOTH_WIN dung trong eeg_within_session_dynamics.ipynb
                       # (lib/within_session_dynamics.py::detect_relaxation_onset) -- neu ve
                       # duong THO thay vi duong DA LAM MUOT, moc onset se roi vao dung luc gia
                       # tri tho dang tut xuong (nhin nhu vo ly du thuat toan dung).


def extract_relax_onset_timeseries(participant_key, study_day, file_name):
    # KHONG downsample (khac bieu do band power) -- doan vuot nguong co the chi keo dai
    # ~6-8s (sustain_n=3 epoch), bin rong se pha loang mat dung doan do khi hien thi, du
    # thuat toan phat hien dung o do phan giai goc. So diem toi da ~900 (session dai nhat),
    # 1 duong duy nhat -- khong dang lo ve kich thuoc file.
    med = eeg_epoch[
        (eeg_epoch["participant_key"] == participant_key) &
        (eeg_epoch["study_day"] == study_day) &
        (eeg_epoch["File"] == file_name) &
        (eeg_epoch["phase"] == "MEDITATION")
    ].sort_values("epoch_idx")
    if med.empty:
        return None

    alpha_rel = ((med["AF3_alpha_power_rel"] + med["AF4_alpha_power_rel"]) / 2).to_numpy()
    alpha_rel_smoothed = np.convolve(alpha_rel, np.ones(ONSET_SMOOTH_WIN) / ONSET_SMOOTH_WIN, mode="same")
    t = np.arange(len(alpha_rel)) * EPOCH_SEC

    row = onset_lookup.loc[file_name] if file_name in onset_lookup.index else None
    threshold = float(row["baseline_alpha_rel"]) if row is not None and pd.notna(row["baseline_alpha_rel"]) else None
    latency = float(row["onset_latency_sec"]) if row is not None and pd.notna(row["onset_latency_sec"]) else None
    reached = bool(row["onset_reached"]) if row is not None and pd.notna(row["onset_reached"]) else None

    return {
        "t": [round(v, 1) for v in t.tolist()],
        "alpha_rel": [round(v, 5) for v in alpha_rel_smoothed.tolist()],
        "threshold": round(threshold, 5) if threshold is not None else None,
        "latency_sec": latency, "reached": reached,
    }

sample = extract_relax_onset_timeseries(*sessions_scope.iloc[1])
print(None if sample is None else {k: (len(v) if isinstance(v, list) else v) for k, v in sample.items()})

## 1c. Chuỗi buồn ngủ (drowsy episode) — theta/alpha ratio, ngưỡng lỏng theo baseline riêng

**Vẫn ở mức THỬ NGHIỆM** (chưa có ground-truth đầy đủ để tính độ nhạy/đặc hiệu chính xác). Ngưỡng = 2× median theta/alpha lúc `REST_BEFORE`
CỦA CHÍNH BUỔI ĐÓ (không dùng ngưỡng tuyệt đối cố định — đã thử và gây âm tính giả với người có
baseline thấp sẵn). Chỉ tiêu chính: **chuỗi epoch liên tục dài nhất** vượt ngưỡng, không phải
trung bình cả đoạn — bắt được cả episode ngắn giữa buổi.

In [ ]:
DROWSY_REL_MULTIPLIER = 2.0
DROWSY_SMOOTH_WIN = 3  # phai khop chinh xac smooth_win truyen vao detect_drowsy_episodes() ben
                        # duoi -- cung ly do nhu ONSET_SMOOTH_WIN o tren, tranh ve duong THO
                        # trong khi quyet dinh dua tren duong DA LAM MUOT.


def extract_drowsiness_timeseries(participant_key, study_day, file_name):
    # KHONG downsample -- cung ly do voi extract_relax_onset_timeseries: doan vuot nguong co
    # the rat ngan, bin rong se lam sai lech vi tri hien thi so voi thuat toan phat hien.
    sub = eeg_epoch[
        (eeg_epoch["participant_key"] == participant_key) &
        (eeg_epoch["study_day"] == study_day) &
        (eeg_epoch["File"] == file_name)
    ]
    rest_before = sub[sub["phase"] == "REST_BEFORE"]
    med = sub[sub["phase"] == "MEDITATION"].sort_values("epoch_idx")
    if len(rest_before) < 3 or med.empty:
        return None

    theta_alpha_rb = ((rest_before["AF3_theta_power_rel"] + rest_before["AF4_theta_power_rel"]) / 2) / \
                     ((rest_before["AF3_alpha_power_rel"] + rest_before["AF4_alpha_power_rel"]) / 2)
    baseline = float(theta_alpha_rb.median())

    theta_alpha_med = ((med["AF3_theta_power_rel"] + med["AF4_theta_power_rel"]) / 2) / \
                      ((med["AF3_alpha_power_rel"] + med["AF4_alpha_power_rel"]) / 2)
    values = theta_alpha_med.to_numpy()
    values_smoothed = np.convolve(values, np.ones(DROWSY_SMOOTH_WIN) / DROWSY_SMOOTH_WIN, mode="same")
    t = np.arange(len(values)) * EPOCH_SEC

    res = detect_drowsy_episodes(values, baseline, epoch_sec=EPOCH_SEC,
                                 smooth_win=DROWSY_SMOOTH_WIN, rel_multiplier=DROWSY_REL_MULTIPLIER)
    run_start_sec = res["longest_run_epoch_idx"] * EPOCH_SEC if res["longest_run_epoch_idx"] is not None else None

    return {
        "t": [round(v, 1) for v in t.tolist()],
        "theta_alpha_ratio": [round(v, 4) for v in values_smoothed.tolist()],
        "threshold": round(res["threshold"], 4),
        "pct_epochs_drowsy": round(res["pct_epochs_drowsy"], 1),
        "longest_run_start_sec": run_start_sec,
        "longest_run_sec": res["longest_run_sec"],
    }

sample = extract_drowsiness_timeseries(*sessions_scope.iloc[1])
print(None if sample is None else {k: (len(v) if isinstance(v, list) else v) for k, v in sample.items()})

## 2. HR/RMSSD toàn phiên — phân giải chi tiết (cửa sổ 30 giây, tính lại từ raw)

Khác bản đầu (chỉ dùng 120s window có sẵn, chỉ cho MEDITATION): ở đây tính
lại trực tiếp từ `ppg_raw.edf` + `event_labels_block.csv` bằng
`lib/ppg_hrv.py`, chia **cả 3 phase** (REST_BEFORE/MEDITATION/REST_AFTER)
thành cửa sổ 30 giây liên tiếp — độ phân giải thời gian gấp ~4 lần bản cũ,
và có luôn REST_BEFORE/REST_AFTER thay vì chỉ 1 điểm trung bình. RMSSD/HR
miền thời gian không cần tối thiểu 60s như LF/HF (`ppg_hrv.py::MIN_DURATION_FOR_FREQ_DOMAIN_S`
chỉ áp dụng cho phổ tần số) nên 30s vẫn cho ước lượng dùng được.

In [ ]:
import sys

LIB_DIR = REPO_ROOT / "lib"
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))
import mne
import ppg_hrv
mne.set_log_level("ERROR")

EEG_FS_EVENTS = 244.0  # sample rate cua latency trong event_labels_block.csv (xem lib/event_labels.py)
PPG_WINDOW_S = 30.0    # do phan giai (truoc la 120s, chi MEDITATION)


def find_session_dir(participant_key, study_day):
    session_id = session_id_lookup.get((participant_key, study_day))
    if session_id is None:
        return None
    d = LAYER2_DIR / f"participant={participant_key}" / f"study_day={study_day:02d}" / f"session_id={session_id}"
    return d if d.exists() else None


def extract_ppg_timeseries(participant_key, study_day, window_s=PPG_WINDOW_S):
    """HR/RMSSD moi 30s, ca 3 phase, tinh lai tu ppg_raw.edf (khong dung ban 120s co san)."""
    session_dir = find_session_dir(participant_key, study_day)
    if session_dir is None:
        return None
    ppg_path = session_dir / "ppg_raw.edf"
    event_path = session_dir / "event_labels_block.csv"
    if not ppg_path.exists() or not event_path.exists():
        return None

    raw = mne.io.read_raw_edf(ppg_path, preload=True, verbose=False)
    ppg = raw.get_data()[0]
    fs = float(raw.info["sfreq"])
    events = pd.read_csv(event_path)

    points = []
    for _, ev in events.iterrows():
        phase = ev["description"]
        if phase not in PHASE_ORDER:
            continue
        start_s = ev["latency"] / EEG_FS_EVENTS
        end_s = (ev["latency"] + ev["duration_samples"]) / EEG_FS_EVENTS
        for w_start, w_end in ppg_hrv.split_windows(start_s, end_s, window_s=window_s):
            ppg_start = max(0, int(round(w_start * fs)))
            ppg_end = min(len(ppg), int(round(w_end * fs)))
            segment = ppg[ppg_start:ppg_end]
            metrics = ppg_hrv.analyze_segment(segment, fs)
            points.append({
                "t": (w_start + w_end) / 2.0,
                "phase": PHASE_ORDER.index(phase),
                "hr": metrics["hr_mean_bpm"],
                "rmssd": metrics["rmssd_ms"],
            })

    if not points:
        return None
    points.sort(key=lambda p: p["t"])

    hr_vals = [p["hr"] for p in points if pd.notna(p["hr"])]
    rmssd_vals = [p["rmssd"] for p in points if pd.notna(p["rmssd"])]
    if not hr_vals and not rmssd_vals:
        return None

    return {
        "t": [round(p["t"], 1) for p in points],
        "phase": [p["phase"] for p in points],
        "hr": [round(p["hr"], 1) if pd.notna(p["hr"]) else None for p in points],
        "rmssd": [round(p["rmssd"], 1) if pd.notna(p["rmssd"]) else None for p in points],
        "hr_max": round(max(hr_vals), 1) if hr_vals else None,
        "hr_min": round(min(hr_vals), 1) if hr_vals else None,
        "rmssd_max": round(max(rmssd_vals), 1) if rmssd_vals else None,
        "rmssd_min": round(min(rmssd_vals), 1) if rmssd_vals else None,
        "window_s": window_s,
    }

print(extract_ppg_timeseries("P_A009", 6))

## 2b. Nhịp thở theo thời gian (thử nghiệm) — cửa sổ 75 giây

PPG-derived respiration (baseline-wander, xem `lib/ppg_resp.py`) — MỚI, chưa có cảm biến hô hấp
tham chiếu để đối chiếu độ chính xác. Dùng cửa sổ **75s** (rộng hơn 30s của HR/RMSSD) vì mỗi nhịp
thở dài ~2-10s, cửa sổ 30s chỉ chứa 3-15 nhịp — quá ít để ước lượng rate/CV ổn định cho hiển thị
từng điểm một trên dashboard cá nhân (khác với dùng tổng hợp qua nhiều session ở mức thống kê,
nơi nhiễu tự triệt tiêu).

In [ ]:
import ppg_resp
import importlib
importlib.reload(ppg_resp)

RESP_WINDOW_S = 75.0


def extract_resp_timeseries(participant_key, study_day, window_s=RESP_WINDOW_S):
    """Nhip tho moi 75s, ca 3 phase, tinh lai tu ppg_raw.edf (cung file voi HR/RMSSD o tren,
    chi khac do rong cua so va ham phan tich)."""
    session_dir = find_session_dir(participant_key, study_day)
    if session_dir is None:
        return None
    ppg_path = session_dir / "ppg_raw.edf"
    event_path = session_dir / "event_labels_block.csv"
    if not ppg_path.exists() or not event_path.exists():
        return None

    raw = mne.io.read_raw_edf(ppg_path, preload=True, verbose=False)
    ppg = raw.get_data()[0]
    fs = float(raw.info["sfreq"])
    events = pd.read_csv(event_path)

    points = []
    for _, ev in events.iterrows():
        phase = ev["description"]
        if phase not in PHASE_ORDER:
            continue
        start_s = ev["latency"] / EEG_FS_EVENTS
        end_s = (ev["latency"] + ev["duration_samples"]) / EEG_FS_EVENTS
        for w_start, w_end in ppg_hrv.split_windows(start_s, end_s, window_s=window_s):
            ppg_start = max(0, int(round(w_start * fs)))
            ppg_end = min(len(ppg), int(round(w_end * fs)))
            segment = ppg[ppg_start:ppg_end]
            metrics = ppg_resp.analyze_segment_resp(segment, fs)
            points.append({
                "t": (w_start + w_end) / 2.0,
                "phase": PHASE_ORDER.index(phase),
                "resp_rate": metrics["resp_rate_bpm"],
            })

    if not points:
        return None
    points.sort(key=lambda p: p["t"])

    resp_vals = [p["resp_rate"] for p in points if pd.notna(p["resp_rate"])]
    if not resp_vals:
        return None

    return {
        "t": [round(p["t"], 1) for p in points],
        "phase": [p["phase"] for p in points],
        "resp_rate": [round(p["resp_rate"], 1) if pd.notna(p["resp_rate"]) else None for p in points],
        "resp_rate_max": round(max(resp_vals), 1),
        "resp_rate_min": round(min(resp_vals), 1),
        "window_s": window_s,
    }

print(extract_resp_timeseries("P_A009", 6))

## 3. HbO/HbR theo thời gian — bản ĐÃ XỬ LÝ (motion-corrected, khớp số liệu chính thức)

Trước dùng thẳng `fnirs_hbo_hbr.csv` (Layer2 v1, CHƯA motion-correction).
Giờ gọi lại đúng pipeline chính thức `lib/fnirs_pipeline_v2.py` (cùng module
`fnirs_hemo_processing_v2.ipynb` dùng để tính `fnirs_hemo_by_phase_v2.csv`):

`load_session_data()` (ưu tiên `fnirs_raw.csv` raw intensity, fallback
`fnirs_hbo_hbr.csv`) → `preprocess_continuous_session()` (OD → **TDDR motion
correction** → Modified Beer-Lambert → lowpass 0.2Hz → robust linear
detrend, chạy 1 lần cho TOÀN session, đúng config trong
`fnirs_hemo_processing_v2.ipynb`). Kết quả này khớp với số liệu thống kê
chính thức trong `session_metadata_survey_features.csv`, khác bản v1 trước
đây.

⚠️ Vẫn là "block dài nhất" đã xuất sẵn ở Layer 2 (có thể không phủ hết toàn
bộ session nếu có đoạn bị cắt do artifact) — giới hạn từ khâu ingestion,
không phải do bước xử lý này.

In [ ]:
import fnirs_pipeline_v2 as fp

FNIRS_CONFIG = {
    "optical_sampling_rate_hz": 100.0,
    "source_detector_distance_cm": 3.0,
    "age_years": 28,
    "motion_correction_method": "TDDR",
    "motion_correction_on_concentration_fallback": True,
    "lowpass_hz": 0.2,
    "highpass_hz": None,
    "detrend_method": "robust_linear_whole_session",
}


def extract_fnirs_timeseries(participant_key, study_day, max_points=150):
    session_dir = find_session_dir(participant_key, study_day)
    if session_dir is None:
        return None
    fnirs_raw_path = session_dir / "fnirs_raw.csv"
    fallback_path = session_dir / "fnirs_hbo_hbr.csv"
    event_path = session_dir / "event_labels_block.csv"

    loaded = fp.load_session_data(
        fnirs_raw_path if fnirs_raw_path.exists() else None,
        fallback_path if fallback_path.exists() else None,
        event_path if event_path.exists() else None,
        optical_fs_expected=FNIRS_CONFIG["optical_sampling_rate_hz"],
    )
    if loaded["input_signal_stage"] is None or loaded["n_samples"] < 10:
        return None

    processed = fp.preprocess_continuous_session(loaded, FNIRS_CONFIG)
    hbo, hbr = processed["hbo_uM"], processed["hbr_uM"]
    t = loaded["time_seconds"]
    if len(t) != len(hbo):
        # bien do dai lech hiem gap (vd loi filter) -- bo qua session nay thay vi ghep sai
        return None

    t_ds = downsample_bin(t.tolist(), max_points)
    hbo_ds = downsample_bin(hbo.tolist(), max_points)
    hbr_ds = downsample_bin(hbr.tolist(), max_points)
    return {
        "t": [round(v, 1) for v in t_ds],
        "hbo": [round(v, 4) for v in hbo_ds],
        "hbr": [round(v, 4) for v in hbr_ds],
        "motion_correction": processed["motion_correction_method"],
        "input_signal_stage": processed["input_signal_stage"],
    }

sample = extract_fnirs_timeseries("P_A011", 5)
print(None if sample is None else {k: len(v) if isinstance(v, list) else v for k, v in sample.items()})

## 4. Chạy cho toàn bộ session nhóm can thiệp (có thiết bị), gom theo participant → study_day

In [ ]:
result = {}
n_eeg = n_ppg = n_resp = n_fnirs = n_onset = n_drowsy = n_start_time = 0
for participant_key, study_day, file_name in sessions_scope.itertuples(index=False):
    if not participant_key.startswith("P_A"):
        continue
    entry = {}
    start_time = start_time_lookup.get((participant_key, study_day))
    if start_time is not None:
        entry["start_time"] = start_time  # "YYYY-MM-DD HH:MM:SS", GMT+7 -- chi hien thi, khong tinh toan
        n_start_time += 1
    eeg_ts = extract_eeg_timeseries(participant_key, study_day, file_name)
    if eeg_ts is not None:
        entry["eeg"] = eeg_ts
        n_eeg += 1
    onset_ts = extract_relax_onset_timeseries(participant_key, study_day, file_name)
    if onset_ts is not None:
        entry["relax_onset"] = onset_ts
        n_onset += 1
    drowsy_ts = extract_drowsiness_timeseries(participant_key, study_day, file_name)
    if drowsy_ts is not None:
        entry["drowsiness"] = drowsy_ts
        n_drowsy += 1
    ppg_ts = extract_ppg_timeseries(participant_key, study_day)
    if ppg_ts is not None:
        entry["ppg"] = ppg_ts
        n_ppg += 1
    resp_ts = extract_resp_timeseries(participant_key, study_day)
    if resp_ts is not None:
        entry["resp"] = resp_ts
        n_resp += 1
    fnirs_ts = extract_fnirs_timeseries(participant_key, study_day)
    if fnirs_ts is not None:
        entry["fnirs"] = fnirs_ts
        n_fnirs += 1
    if entry:
        result.setdefault(participant_key, {})[int(study_day)] = entry

out_path = OUT_DIR / "session_timeseries.json"
with open(out_path, "w") as f:
    json.dump(result, f, separators=(",", ":"))

print(f"Da luu: {out_path}")
print(f"Session co EEG timeseries: {n_eeg} | relax_onset: {n_onset} | drowsiness: {n_drowsy} | PPG: {n_ppg} | Resp: {n_resp} | fNIRS: {n_fnirs} | start_time: {n_start_time}")
print(f"So participant: {len(result)}")
import os
print(f"Kich thuoc file: {os.path.getsize(out_path) / 1024:.0f} KB")

## Tổng kết

- Không phải session nào cũng có đủ cả 3 loại time series (fNIRS đặc biệt
  hay thiếu — session bị QC fail hoặc không xuất được `fnirs_hbo_hbr.csv`).
- HbO/HbR dùng bản **CHƯA motion-correction** (v1) — khác với số liệu thống
  kê chính thức trong `session_metadata_survey_features.csv` (dùng bản v2 đã
  TDDR+MBLL). Chỉ dùng cho trực quan hoá xu hướng, không dùng để so sánh số.
- Nhịp thở (`resp`) là ước lượng PPG-derived MỚI (thử nghiệm), cửa sổ 75s riêng, tách khỏi `ppg`.
